# Exploration initiale de la base SQLite

## Objectif

Ce notebook documente une première exploration de la base SQLite brute située dans `data/raw/risk_monitor_dataset.sqlite`.

L'objectif est de comprendre la structure des données et d'identifier des points potentiels de qualité avant toute étape de nettoyage, de transformation ou de scoring.

Aucune correction n'est appliquée dans ce notebook. Les cellules ci-dessous servent uniquement à observer les tables, leurs colonnes, leurs valeurs manquantes, les doublons possibles, les timestamps et les relations apparentes entre tables.

## 1. Chargement des dépendances et connexion SQLite

Cette section charge les bibliothèques nécessaires et prépare la connexion à la base SQLite brute.

In [1]:
from pathlib import Path
import sqlite3
import pandas as pd

In [ ]:
DB_PATH = Path("../data/raw/risk_monitor_dataset.sqlite")

if not DB_PATH.exists():
    raise FileNotFoundError(f"Base SQLite introuvable : {DB_PATH}")

connection = sqlite3.connect(DB_PATH)
connection

FileNotFoundError: Base SQLite introuvable : data\raw\risk_monitor_dataset.sqlite

## 2. Liste des tables

Cette section récupère les tables déclarées dans la base SQLite, sans hypothèse sur leur rôle métier.

In [ ]:
tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
    """,
    connection,
)

tables

In [ ]:
table_names = tables["name"].tolist()
table_names

## 3. Aperçu de chaque table

Pour chaque table, on affiche le nombre de lignes, le schéma SQLite et les 5 premières lignes. Cette étape reste descriptive.

In [ ]:
def quote_identifier(identifier: str) -> str:
    return '"' + identifier.replace('"', '""') + '"'


def inspect_table(table_name: str) -> None:
    quoted_table_name = quote_identifier(table_name)

    row_count = pd.read_sql_query(
        f"SELECT COUNT(*) AS row_count FROM {quoted_table_name};",
        connection,
    )
    schema = pd.read_sql_query(f"PRAGMA table_info({quoted_table_name});", connection)
    preview = pd.read_sql_query(f"SELECT * FROM {quoted_table_name} LIMIT 5;", connection)

    print("=" * 80)
    print(f"Table : {table_name}")
    display(row_count)
    display(schema[["name", "type", "notnull", "pk"]])
    display(preview)


for table_name in table_names:
    inspect_table(table_name)

## 4. Analyse des valeurs manquantes

Cette section mesure les valeurs manquantes par table et par colonne. Elle ne remplace, ne supprime et ne corrige aucune valeur.

In [ ]:
for table_name in table_names:
    quoted_table_name = quote_identifier(table_name)
    df = pd.read_sql_query(f"SELECT * FROM {quoted_table_name};", connection)

    missing_summary = pd.DataFrame(
        {
            "column": df.columns,
            "missing_count": df.isna().sum().values,
            "missing_rate": df.isna().mean().values,
        }
    ).sort_values("missing_count", ascending=False)

    print("=" * 80)
    print(f"Table : {table_name}")
    display(missing_summary)

## 5. Détection des doublons

Cette section compte les lignes exactement dupliquées dans chaque table. Elle ne déduit pas encore si ces doublons sont problématiques.

In [ ]:
duplicate_summaries = []

for table_name in table_names:
    quoted_table_name = quote_identifier(table_name)
    df = pd.read_sql_query(f"SELECT * FROM {quoted_table_name};", connection)

    duplicate_summaries.append(
        {
            "table": table_name,
            "row_count": len(df),
            "exact_duplicate_rows": int(df.duplicated().sum()),
        }
    )

pd.DataFrame(duplicate_summaries)

## 6. Valeurs catégorielles incohérentes

Cette section aide à repérer les colonnes textuelles avec des modalités proches, rares ou inattendues. Aucun regroupement ou remplacement n'est effectué.

In [ ]:
MAX_UNIQUE_VALUES_TO_DISPLAY = 30

for table_name in table_names:
    quoted_table_name = quote_identifier(table_name)
    df = pd.read_sql_query(f"SELECT * FROM {quoted_table_name};", connection)
    text_columns = df.select_dtypes(include="object").columns

    print("=" * 80)
    print(f"Table : {table_name}")

    if len(text_columns) == 0:
        print("Aucune colonne textuelle détectée par pandas.")
        continue

    for column in text_columns:
        value_counts = df[column].value_counts(dropna=False).head(MAX_UNIQUE_VALUES_TO_DISPLAY)
        print(f"\nColonne : {column}")
        display(value_counts.to_frame("count"))

## 7. Problèmes de timestamps

Cette section cible les colonnes dont le nom suggère une date ou un timestamp. Les conversions sont exploratoires et servent uniquement à repérer des valeurs non interprétables par pandas.

In [ ]:
TIMESTAMP_KEYWORDS = ["date", "time", "timestamp", "created", "updated"]

timestamp_summaries = []

for table_name in table_names:
    quoted_table_name = quote_identifier(table_name)
    df = pd.read_sql_query(f"SELECT * FROM {quoted_table_name};", connection)

    candidate_columns = [
        column
        for column in df.columns
        if any(keyword in column.lower() for keyword in TIMESTAMP_KEYWORDS)
    ]

    for column in candidate_columns:
        parsed = pd.to_datetime(df[column], errors="coerce")
        timestamp_summaries.append(
            {
                "table": table_name,
                "column": column,
                "non_missing_values": int(df[column].notna().sum()),
                "unparsed_values": int((df[column].notna() & parsed.isna()).sum()),
                "min_parsed": parsed.min(),
                "max_parsed": parsed.max(),
            }
        )

pd.DataFrame(timestamp_summaries)

## 8. Analyse des champs `status` non documentés

Cette section recense les colonnes dont le nom contient `status` et affiche leurs distributions observées. Aucune signification métier n'est attribuée aux valeurs.

In [ ]:
for table_name in table_names:
    quoted_table_name = quote_identifier(table_name)
    df = pd.read_sql_query(f"SELECT * FROM {quoted_table_name};", connection)
    status_columns = [column for column in df.columns if "status" in column.lower()]

    if not status_columns:
        continue

    print("=" * 80)
    print(f"Table : {table_name}")

    for column in status_columns:
        print(f"\nChamp status : {column}")
        display(df[column].value_counts(dropna=False).to_frame("count"))

## 9. Vérification de cohérence entre les tables

Cette section prépare des contrôles simples sur les clés potentielles et les colonnes communes. Elle ne suppose pas encore de relations métier ou de contraintes de clé étrangère.

In [ ]:
schemas = {}

for table_name in table_names:
    quoted_table_name = quote_identifier(table_name)
    schema = pd.read_sql_query(f"PRAGMA table_info({quoted_table_name});", connection)
    schemas[table_name] = schema

column_locations = []
for table_name, schema in schemas.items():
    for column in schema["name"]:
        column_locations.append({"column": column, "table": table_name})

column_locations = pd.DataFrame(column_locations)

common_columns = (
    column_locations.groupby("column")["table"]
    .apply(list)
    .reset_index(name="tables")
)
common_columns["table_count"] = common_columns["tables"].str.len()
common_columns = common_columns.sort_values(["table_count", "column"], ascending=[False, True])

common_columns[common_columns["table_count"] > 1]

In [ ]:
foreign_keys = []

for table_name in table_names:
    quoted_table_name = quote_identifier(table_name)
    table_foreign_keys = pd.read_sql_query(f"PRAGMA foreign_key_list({quoted_table_name});", connection)
    if not table_foreign_keys.empty:
        table_foreign_keys.insert(0, "source_table", table_name)
        foreign_keys.append(table_foreign_keys)

if foreign_keys:
    pd.concat(foreign_keys, ignore_index=True)
else:
    pd.DataFrame(columns=["source_table", "id", "seq", "table", "from", "to", "on_update", "on_delete", "match"])

## 10. Décisions de nettoyage

Cette section est volontairement laissée vide à ce stade.

Elle sera complétée après revue des observations ci-dessus, avec des décisions explicites et justifiées.

In [ ]:
# À compléter plus tard.
# Ne pas appliquer de nettoyage dans cette version du notebook.

## Fermeture de la connexion

Fermer la connexion SQLite une fois l'exploration terminée.

In [ ]:
connection.close()